In [ ]:
import torch
import torch.nn as nn
from torch.nn.functional import log_softmax, pad
import copy
import math
import time
import random
from torch.optim.lr_scheduler import LambdaLR
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

In [ ]:
class EncoderDecoder(nn.Module):
  """
  a standard Encoder-Decoder architecture. base for this and many
  other models.
  """

  def __init__(self, encoder, decoder, src_embed, tgt_embed, generator):
    super().__init__()
    self.encoder = encoder
    self.decoder = decoder
    self.src_embed = src_embed
    self.tgt_embed = tgt_embed
    self.generator = generator

  def forward(self, src, tgt, src_mask, tgt_mask):
    "take in and process masked src and target sequences."
    return self.decode(self.encode(src, src_mask), src_mask, tgt, tgt_mask)

  def encode(self, src, src_mask):
    return self.encoder(self.src_embed(src), src_mask)

  def decode(self, memory, src_mask, tgt, tgt_mask):
    return self.decoder(self.tgt_embed(tgt), memory, src_mask, tgt_mask)
    
class Generator(nn.Module):
  "define standard linear + softmax generation step"

  def __init__(self, d_model, vocab):
    super().__init__()
    self.proj = nn.Linear(d_model, vocab)

  def forward(self, x):
    return log_softmax(self.proj(x), dim=-1)

In [ ]:
def clones(module, N):
  "produce N identical layers."
  return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])

class Encoder(nn.Module):
  "core encoder is a stack of N layers"
  
  def __init__(self, layer, N):
    super().__init__()
    self.layers = clones(layer, N)
    self.norm = LayerNorm(layer.size)

  def forward(self, x, mask):
    "pass the input (and mask) through each layer in turn."
    for layer in self.layers:
      x = layer(x, mask)
    return self.norm(x)
  
class LayerNorm(nn.Module):
  "construct a layernorm module."
  
  def __init__(self, features, eps=1e-6):
    super().__init__()
    self.a_2 = nn.Parameter(torch.ones(features))
    self.b_2 = nn.Parameter(torch.zeros(features))
    self.eps = eps

  def forward(self, x):
    mean = x.mean(-1, keepdim=True)
    std = x.std(-1, keepdim=True)
    return self.a_2 * (x - mean) / (std + self.eps) + self.b_2

In [ ]:
class SublayerConnection(nn.Module):
  """
  a residual connection followed by a layer norm.
  note for code simplicity the norm is first as opposed to last.
  """

  def __init__(self, size, dropout):
    super().__init__()
    self.norm = LayerNorm(size)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x, sublayer):
    "apply residual connection to any sublayer with the same size."
    return x + self.dropout(sublayer(self.norm(x)))

In [ ]:
class EncoderLayer(nn.Module):
  "encoder is made up of self-attn and feed forward (defined below)"

  def __init__(self, size, self_attn, feed_forward, dropout):
    super().__init__()
    self.self_attn = self_attn
    self.feed_forward = feed_forward
    self.sublayer = clones(SublayerConnection(size, dropout), 2)
    self.size = size

  def forward(self, x, mask):
    x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, mask))
    return self.sublayer[1](x, self.feed_forward)

In [ ]:
class Decoder(nn.Module):
  "generic N layer decoder with masking."

  def __init__(self, layer, N):
    super().__init__()
    self.layers = clones(layer, N)
    self.norm = LayerNorm(layer.size)

  def forward(self, x, memory, src_mask, tgt_mask):
    for layer in self.layers:
      x = layer(x, memory, src_mask, tgt_mask)
    return self.norm(x)

In [ ]:
class DecoderLayer(nn.Module):
  "decoder is made of self-attn, src-attn, and feed forward"

  def __init__(self, size, self_attn, src_attn, feed_forward, dropout):
    super().__init__()
    self.size = size
    self.self_attn = self_attn
    self.src_attn = src_attn
    self.feed_forward = feed_forward
    self.sublayer = clones(SublayerConnection(size, dropout), 3)

  def forward(self, x, memory, src_mask, tgt_mask):
    m = memory
    x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, tgt_mask))
    x = self.sublayer[1](x, lambda x: self.src_attn(x, m, m, src_mask))
    return self.sublayer[2](x, self.feed_forward)

In [ ]:
def subsequent_mask(size, device=None):
  "mask out subsequent positions."
  mask = torch.tril(
    torch.ones(
      (1, 1, size, size),
      dtype=torch.bool,
      device=device,
    )
  )
  
  return mask

print(subsequent_mask(5))
  

In [ ]:
def attention(query, key, value, mask=None, dropout=None):
  "compute 'scaled dot product attention'"
  d_k = query.size(-1)

  scores = (query @ key.transpose(-2, -1)) / math.sqrt(d_k)

  if mask is not None:
    scores.masked_fill_(~mask, -torch.inf)

  attn = scores.softmax(dim=-1)
  if dropout is not None:
    attn = dropout(attn)
  
  output = attn @ value

  return output, attn


In [ ]:
class MultiHeadedAttention(nn.Module):
  def __init__(self, d_model, num_heads, dropout=0.1,):
    "take in model size and number of heads."
    super().__init__()
    assert d_model % num_heads == 0
    
    self.d_model = d_model
    self.num_heads = num_heads
    self.head_dim = d_model // num_heads

    # qkv projections
    self.q_proj = nn.Linear(d_model, d_model)
    self.k_proj = nn.Linear(d_model, d_model)
    self.v_proj = nn.Linear(d_model, d_model)

    # output projection
    self.out_proj = nn.Linear(d_model, d_model)
    
    self.dropout = nn.Dropout(dropout)

  def forward(self, query, key, value, mask=None):
    nbatches = query.size(0)

    q = self.q_proj(query)
    k = self.k_proj(key)
    v = self.v_proj(value)

    q = q.reshape(nbatches, -1, self.num_heads, self.head_dim).transpose(1, 2)
    k = k.reshape(nbatches, -1, self.num_heads, self.head_dim).transpose(1, 2)
    v = v.reshape(nbatches, -1, self.num_heads, self.head_dim).transpose(1, 2)
    
    x, attn = attention(q, k, v, mask=mask, dropout=self.dropout)
    
    # optional: save attention map for visualization
    self.attn = attn

    x = x.transpose(1, 2).reshape(nbatches, -1, self.d_model)
    
    x = self.out_proj(x)

    return x

In [ ]:
class PositionwiseFeedForward(nn.Module):
  "implements FFN equation."

  def __init__(self, d_model, d_ff, dropout=0.1):
    super().__init__()
    self.w_1 = nn.Linear(d_model, d_ff)
    self.w_2 = nn.Linear(d_ff, d_model)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    return self.w_2(self.dropout(self.w_1(x).relu()))

In [ ]:
class Embeddings(nn.Module):
  def __init__(self, vocab_size, d_model):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, d_model)
    self.scale = math.sqrt(d_model)
    self.d_model = d_model

  def forward(self, x):
    return self.embedding(x) * self.scale

In [ ]:
class PositionalEncoding(nn.Module):

  def __init__(self, d_model, dropout=0.1, max_len=5000):
    super().__init__()
    self.dropout = nn.Dropout(p=dropout)

    # compute the positional encodings once in log space.
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(max_len).unsqueeze(1)

    div_term = torch.exp(
      torch.arange(0, d_model, 2)
      * (-math.log(10000.0) / d_model)
    )
    
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    pe = pe.unsqueeze(0)
    
    # not a parameter
    self.register_buffer("pe", pe)

  def forward(self, x):
    x = x + self.pe[:, : x.size(1)]
    return self.dropout(x)

In [ ]:
def make_model(src_vocab, tgt_vocab, N=6, d_model=512, d_ff=2048, num_heads=8, dropout=0.1):
  "helper: construct a model from hyperparameters."
  c = copy.deepcopy
  attn = MultiHeadedAttention(d_model, num_heads)
  ff = PositionwiseFeedForward(d_model, d_ff, dropout)
  position = PositionalEncoding(d_model, dropout)
  model = EncoderDecoder(
    Encoder(EncoderLayer(d_model, c(attn), c(ff), dropout), N),
    Decoder(DecoderLayer(d_model, c(attn), c(attn), c(ff), dropout), N),
    nn.Sequential(Embeddings(src_vocab, d_model), c(position)),
    nn.Sequential(Embeddings(tgt_vocab, d_model), c(position)),
    Generator(d_model, tgt_vocab)
  )

  # this was important from their code.
  # xavier initialization
  for p in model.parameters():
    if p.dim() > 1:
      nn.init.xavier_uniform_(p)
  return model

In [ ]:
def inference_test():
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  
  model = make_model(11, 11, 2).to(device)
  model.eval()
  
  src = torch.tensor(
    [[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]],
    dtype=torch.long,
    device=device
  )

  src_mask = torch.ones(
    (1, 1, 1, src.size(1)),
    dtype=torch.bool,
    device=device,
  )

  memory = model.encode(src, src_mask)

  ys = torch.zeros(
    (1, 1),
    dtype=torch.long,
    device=device
  )

  for _ in range(src.size(1) - 1):
    tgt_mask = subsequent_mask(ys.size(1), device=device)
    out = model.decode(memory, src_mask, ys, tgt_mask)
    logits = model.generator(out[:, -1])
    next_token = logits.argmax(dim=-1)
    ys = torch.cat(
      [ys, next_token.unsqueeze(1)],
      dim=1,
    )

  print("example untrained model prediction:", ys)

def run_tests():
  for _ in range(10):
    inference_test()

run_tests()

In [ ]:
def make_std_mask(tgt, pad):
  """
  decoder self-attention mask.

  combines:
  1. padding mask
  2. causal mask

  tgt: 
    (batch_size, seq_len)

  return:
    (batch_size, 1, seq_len, seq_len)
  """

  _, seq_len = tgt.shape
  device = tgt.device

  # padding mask shape:
  # (batch_size, 1, 1, seq_len)
  padding_mask = (tgt != pad).unsqueeze(1).unsqueeze(2)

  # causal mask shape:
  # (1, 1, seq_len, seq_len)
  causal_mask = subsequent_mask(seq_len, device=device)
  
  # combine shape:
  # (batch_size, 1, seq_len, seq_len)
  return padding_mask & causal_mask

class Batch:
  """object for holding a batch of data with mask during training."""

  def __init__(self, src, tgt=None, pad=0):  # 0 = <blank>
    self.src = src
    self.src_mask = (src != pad).unsqueeze(1).unsqueeze(2)
    if tgt is not None:
      # decoder input
      self.tgt = tgt[:, :-1]
      # prediction target
      self.tgt_y = tgt[:, 1:]

      # decoder self-attention mask
      self.tgt_mask = make_std_mask(self.tgt, pad)
      
      self.ntokens = (self.tgt_y != pad).sum().item()

In [ ]:
def rate(step, model_size, factor, warmup):
  """
  we have to default the step to 1 for LambdaLR function
  to avoid zero raising to negative power.
  """
  if step == 0:
    step = 1
  return factor * (model_size ** (-0.5) * min(step ** (-0.5), step * warmup ** (-1.5)))

In [ ]:
def example_learning_schedule():
  opts = [
    [512, 1, 4000],  # example 1
    [512, 1, 8000],  # example 2
    [256, 1, 4000],  # example 3
  ]

  dummy_model = torch.nn.Linear(1, 1)
  learning_rates = []

  # we have 3 examples in opts list.
  for idx, example in enumerate(opts):
    # run 20000 epoch for each example
    optimizer = torch.optim.Adam(dummy_model.parameters(), lr=1, betas=(0.9, 0.98), eps=1e-9)
    lr_scheduler = LambdaLR(optimizer=optimizer, lr_lambda=lambda step: rate(step, *example))

    tmp = []
    # take 20K dummy training steps, save the learning rate at each step
    
    for step in range(20000):
      tmp.append(optimizer.param_groups[0]["lr"])
      optimizer.step()
      lr_scheduler.step()
      
    learning_rates.append(tmp)

  learning_rates = torch.tensor(learning_rates)

  return learning_rates


def plot_learning_schedule(learning_rates):
  labels = ["512:4000", "512:8000", "256:4000"]

  plt.figure(figsize=(10, 6))

  for i in range(3):
    plt.plot(range(20000), learning_rates[i], label=labels[i])

  plt.xlabel("step")
  plt.ylabel("learning rate")

  plt.xticks(range(0, 20001, 2000))
  plt.legend()
  plt.grid(True)

  plt.show()


learning_rates = example_learning_schedule()
plot_learning_schedule(learning_rates)

In [ ]:
class LabelSmoothing(nn.Module):
  "implement label smoothing."

  def __init__(self, vocab_size, padding_idx, smoothing=0.0):
    super().__init__()
    self.criterion = nn.KLDivLoss(reduction="sum")
    self.padding_idx = padding_idx
    self.confidence = 1.0 - smoothing
    self.smoothing = smoothing
    self.vocab_size = vocab_size
    self.true_dist = None


  def forward(self, x, target):
    
    assert x.size(1) == self.vocab_size
    
    true_dist = x.detach().clone()
    
    true_dist.fill_(self.smoothing / (self.vocab_size - 2))
    true_dist.scatter_(1, target.unsqueeze(1), self.confidence)
    true_dist[:, self.padding_idx] = 0
    mask = (target == self.padding_idx)
    
    if mask.any():
      true_dist.masked_fill_(mask.unsqueeze(1), 0.0)
      
    self.true_dist = true_dist
    return self.criterion(x, true_dist)
  

In [ ]:
# example of label smoothing.

def example_label_smoothing():
  crit = LabelSmoothing(5, 0, 0.4)
  
  predict = torch.FloatTensor(
    [
      [0, 0.2, 0.7, 0.1, 0],
      [0, 0.2, 0.7, 0.1, 0],
      [0, 0.2, 0.7, 0.1, 0],
      [0, 0.2, 0.7, 0.1, 0],
      [0, 0.2, 0.7, 0.1, 0],
    ]
  )
  
  crit(x=predict.log(), target=torch.LongTensor([2, 1, 0, 3, 3]))
  
  true_dist = crit.true_dist.numpy()

  plt.figure(figsize=(6, 6))

  plt.imshow(true_dist, cmap="viridis")

  plt.colorbar(label="target distribution")

  plt.xticks(range(5))
  plt.yticks(range(5))

  plt.show()


example_label_smoothing()

In [ ]:
def loss(x, crit):
  d = x + 3 * 1
  predict = torch.FloatTensor([[1e-9, x / d, 1 / d, 1 / d, 1 / d]])
  return crit(predict.log(), torch.LongTensor([1])).data


def penalization_visualization():
  crit = LabelSmoothing(5, 0, 0.1)

  steps = list(range(1, 100))
  losses = [loss(x, crit) for x in steps]

  plt.figure(figsize=(8, 5))

  plt.plot(steps, losses, linewidth=2, label="label smoothing loss")

  plt.xlabel("steps")
  plt.ylabel("loss")
  plt.title("penalization of overconfidence")

  plt.xticks(range(0, 101, 10))

  plt.grid(True)
  plt.legend()
  plt.show()

   
penalization_visualization()

In [ ]:
def data_gen(V, batch_size, nbatches, device):
  "generate random data for a src-tgt copy task."
  for _ in range(nbatches):
    data = torch.randint(1, V, size=(batch_size, 10), device=device)
    data[:, 0] = 1
    src = data.clone()
    tgt = data.clone()
    yield Batch(src, tgt, 0)

In [ ]:
class SimpleLossCompute:
  "a simple loss compute and train function."

  def __init__(self, generator, criterion):
    self.generator = generator
    self.criterion = criterion

  def __call__(self, x, y, norm):
    logits = self.generator(x)
    logits = logits.reshape(-1, logits.size(-1))
    y = y.reshape(-1)
    
    loss = self.criterion(logits, y) / norm
        
    return loss / norm

In [ ]:
def greedy_decode(model, src, src_mask, max_len, start_symbol):
  memory = model.encode(src, src_mask)
  ys = torch.tensor([[start_symbol]], dtype=src.dtype, device=src.device)

  for i in range(max_len - 1):
    tgt_mask = subsequent_mask(ys.size(1), device=src.device)
    out = model.decode(memory, src_mask, ys, tgt_mask)
    prob = model.generator(out[:, -1])
    next_word = prob.argmax(dim=-1)
    next_token = next_word.unsqueeze(0)
    ys = torch.cat([ys, next_token], dim=1)
  return ys

In [ ]:
def train_epoch(
  model,
  dataloader,
  loss_compute,
  optimizer,
  scheduler,
  accum_steps=1,
  log_every=40,
):

  model.train()

  total_loss = 0.0
  total_tokens = 0

  start_time = time.time()

  optimizer.zero_grad(set_to_none=True)

  for step, batch in enumerate(dataloader):
    logits = model(batch.src, batch.tgt, batch.src_mask, batch.tgt_mask)
    loss = loss_compute(logits, batch.tgt_y, batch.ntokens)

    # gradient accumulation
    loss = loss / accum_steps

    loss.backward()

    if (step + 1) % accum_steps == 0:
      optimizer.step()
      optimizer.zero_grad(set_to_none=True)
      scheduler.step()

    batch_loss = loss.item() * accum_steps
    total_loss += batch_loss * batch.ntokens
    total_tokens += batch.ntokens

    if step % log_every == 0:
      elapsed = time.time() - start_time
      tokens_per_sec = (total_tokens / elapsed)
      lr = optimizer.param_groups[0]["lr"]

      #print(f"step={step:6d} | loss={batch_loss:8.4f} | tokens/sec={tokens_per_sec:8.1f} | lr={lr:8.2e}")

  return total_loss / total_tokens


In [ ]:
def evaluate_epoch(model, dataloader, loss_compute):
  model.eval()

  total_loss = 0.0
  total_tokens = 0

  for batch in dataloader:
    logits = model(batch.src, batch.tgt, batch.src_mask, batch.tgt_mask)
    loss = loss_compute(logits, batch.tgt_y, batch.ntokens)
    total_loss += loss.item() * batch.ntokens
    total_tokens += batch.ntokens

  return total_loss / total_tokens

In [ ]:
# train the simple copy task.

def example_simple_model():

  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

  print("using device:", device)

  vocab_size = 11

  criterion = LabelSmoothing(
    vocab_size=vocab_size,
    padding_idx=0,
    smoothing=0.0,
  )

  model = make_model(
    vocab_size,
    vocab_size,
    N=2,
  ).to(device)

  optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.5,
    betas=(0.9, 0.98),
    eps=1e-9,
  )

  scheduler = LambdaLR(
    optimizer=optimizer,
    lr_lambda=lambda step: rate(
      step, model_size=model.src_embed[0].d_model, factor=1.0, warmup=400
    ),
  )

  loss_compute = SimpleLossCompute(model.generator, criterion)

  batch_size = 80
  epochs = 20
  nbatches = 20

  for epoch in range(epochs):

    train_loss = train_epoch(
      model=model,
      dataloader=data_gen(vocab_size, batch_size, nbatches, device=device),
      loss_compute=loss_compute,
      optimizer=optimizer,
      scheduler=scheduler,
    )

    val_loss = evaluate_epoch(
      model=model,
      dataloader=data_gen(vocab_size, batch_size, 5, device=device),
      loss_compute=loss_compute,
    )

    print(f"epoch={epoch:6d} | train_loss={train_loss:8.4f} | val_loss={val_loss:8.4f}")

  model.eval()

  src = torch.tensor(
    [[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]],
    dtype=torch.long,
    device=device,
  )

  max_len = src.size(1)

  src_mask = torch.ones(
    (1, 1, 1, max_len),
    dtype=torch.bool,
    device=device,
  )

  prediction = greedy_decode(model, src, src_mask, max_len=max_len, start_symbol=0)
  print(prediction)

example_simple_model()

In [ ]:
def write_lines(path, data):
  with open(path, "w", encoding="utf-8") as f:
    f.writelines(data)

def split_dataset(src_path="data/cmn.txt", seed=42):
  """
  the dataset is downloaded from:
  www.manythings.org/anki/

  specifically:
  Chinese-English sentence pairs (cmn-eng.zip)

  this script splits the cleaned cmn.txt into:
    - train.txt      (80%)
    - dev.txt        (10%)
    - test.txt       (10%)

  additionally create:
    - train_mini.txt
    - dev_mini.txt
    - test_mini.txt

  the mini datasets are small subsets used for debugging and quick experiments.
  """

  random.seed(seed)

  with open(src_path, "r", encoding="utf-8") as f:
    lines = [line for line in f if line.strip()]

  total = len(lines)

  random.shuffle(lines)

  train_end = int(total * 0.8)
  dev_end = train_end + int(total * 0.1)

  train_lines = lines[:train_end]
  dev_lines = lines[train_end:dev_end]
  test_lines = lines[dev_end:]

  write_lines("data/train.txt", train_lines)
  write_lines("data/dev.txt", dev_lines)
  write_lines("data/test.txt", test_lines)


  write_lines("data/train_mini.txt", train_lines[:1000])
  write_lines("data/dev_mini.txt", dev_lines[:200])
  write_lines("data/test_mini.txt", test_lines[:200])


  print(f"total lines : {total}")
  print(f"train lines : {len(train_lines)}")
  print(f"dev lines   : {len(dev_lines)}")
  print(f"test lines  : {len(test_lines)}")

  print()

  print(f"train_mini  : {min(1000, len(train_lines))}")
  print(f"dev_mini    : {min(200, len(dev_lines))}")
  print(f"test_mini   : {min(200, len(test_lines))}")


split_dataset(src_path="data/cmn.txt")

In [ ]:
UNK = 0  # unknown word-id
PAD = 1  # padding word-id

def seq_padding(seqs, pad=PAD):
  """
  pad sequences to same length.
  """
  max_len = max(len(seq) for seq in seqs)
  return np.array([
    seq + [pad] * (max_len - len(seq))
    for seq in seqs
  ])

class TranslationDataset:

  def __init__(self, train_path, dev_path, device):

    self.device = device

    # load raw data
    train_en, train_cn = self.load_data(train_path)
    dev_en, dev_cn = self.load_data(dev_path)

    # build vocab
    self.en_vocab, self.en_id2word = self.build_vocab(train_en)
    self.cn_vocab, self.cn_id2word = self.build_vocab(train_cn)

    # vocab size
    self.src_vocab_size = len(self.en_vocab)
    self.tgt_vocab_size = len(self.cn_vocab)

    # token -> id
    train_en, train_cn = self.tokens_to_ids(train_en, train_cn, self.en_vocab, self.cn_vocab)
    dev_en, dev_cn = self.tokens_to_ids(dev_en, dev_cn, self.en_vocab, self.cn_vocab)

    # build batches
    self.train_en_ids = train_en
    self.train_cn_ids = train_cn

    self.dev_en_ids = dev_en
    self.dev_cn_ids = dev_cn

  def load_data(self, path):
    """
    tokenize the sentence and add BOS/EOS marks(begin of sentence; end of sentence)
    en = [['BOS', 'i', 'love', 'you', 'EOS'], 
          ['BOS', 'me', 'too', 'EOS'], ...]
    cn = [['BOS', '我', '爱', '你', 'EOS'], 
          ['BOS', '我', '也', '是', 'EOS'], ...]
    """
    en_sentences = []
    cn_sentences = []

    with open(path, 'r', encoding='utf-8') as f:
      for line in f:
        en, cn = line.strip().split('\t')

        en_tokens = ["BOS"] + en.lower().split() + ["EOS"]
        cn_tokens = ["BOS"] + list(cn) + ["EOS"]

        en_sentences.append(en_tokens)
        cn_sentences.append(cn_tokens)

    return en_sentences, cn_sentences

  def build_vocab(self, sentences, max_words=50000):
    counter = Counter()

    for sent in sentences:
      for w in sent:
        counter[w] += 1

    vocab = {
      word: idx + 2
      for idx, (word, _) in enumerate(counter.most_common(max_words))
    }

    vocab["UNK"] = UNK
    vocab["PAD"] = PAD

    id2word = {
      idx: word
      for word, idx in vocab.items()
    }

    return vocab, id2word

  def tokens_to_ids(self, en_sentences, cn_sentences, en_vocab, cn_vocab, sort=True):

    en_ids = [
      [en_vocab.get(w, UNK) for w in sent]
      for sent in en_sentences
    ]

    cn_ids = [
      [cn_vocab.get(w, UNK) for w in sent]
      for sent in cn_sentences
    ]

    if sort:
      pairs = list(zip(en_ids, cn_ids))
      pairs.sort(key=lambda x: len(x[0]))
      en_ids, cn_ids = zip(*pairs)

    return list(en_ids), list(cn_ids)

  def build_batches(self, en_ids, cn_ids, batch_size, shuffle=True):
      indices = np.arange(len(en_ids))

      if shuffle:
        np.random.shuffle(indices)

      batches = []

      for start in range(0, len(indices), batch_size):
        batch_idx = indices[start:start + batch_size]
        batch_en = seq_padding([en_ids[i] for i in batch_idx])
        batch_cn = seq_padding([cn_ids[i] for i in batch_idx])
        
        batch_en = torch.tensor(batch_en, dtype=torch.long, device=self.device)
        batch_cn = torch.tensor(batch_cn, dtype=torch.long, device=self.device)
        yield Batch(batch_en, batch_cn, pad=PAD)

  def train_dataloader(self, batch_size):
    return self.build_batches(self.train_en_ids, self.train_cn_ids, batch_size)

  def dev_dataloader(self, batch_size):
    return self.build_batches(self.dev_en_ids, self.dev_cn_ids, batch_size, shuffle=False)
  

In [ ]:
DEBUG = True    # for debug

if DEBUG:          
  train_path = 'data/train_mini.txt'
  dev_path   = 'data/dev_mini.txt'
  model_path = 'save/models/model.pt' 
else:
  train_path = 'data/train.txt'   
  dev_path   = 'data/dev.txt'   
  model_path = 'save/models/large_model.pt'  

def translation_model():

  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

  print("using device:", device)

  data = TranslationDataset(train_path, dev_path, device=device)
  print("src_vocab:", data.src_vocab_size)
  print("tgt_vocab:", data.tgt_vocab_size)

  model = make_model(
    data.src_vocab_size,
    data.tgt_vocab_size,
    N=2
  ).to(device)
  
  criterion = LabelSmoothing(
    vocab_size=data.tgt_vocab_size,
    padding_idx=0,
    smoothing=0.0,
  )

  optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.5,
    betas=(0.9, 0.98),
    eps=1e-9,
  )

  scheduler = LambdaLR(
    optimizer=optimizer,
    lr_lambda=lambda step: rate(
      step, model_size=model.src_embed[0].d_model, factor=1.0, warmup=400
    )
  )

  loss_compute = SimpleLossCompute(model.generator, criterion)

  epochs = 20
  batch_size = 64

  for epoch in range(epochs):

    train_loss = train_epoch(
      model=model,
      dataloader=data.train_dataloader(batch_size),
      loss_compute=loss_compute,
      optimizer=optimizer,
      scheduler=scheduler,
    )

    val_loss = evaluate_epoch(
      model=model,
      dataloader=data.dev_dataloader(batch_size),
      loss_compute=loss_compute,
    )

    print(f"epoch={epoch:6d} | train_loss={train_loss:8.4f} | val_loss={val_loss:8.4f}")

  return model, data

model, data = translation_model()

In [ ]:
def evaluate(dataset, model, num_examples=10):

  model.eval()

  random_indices = np.random.choice(len(dataset.dev_en_ids), num_examples)

  for idx in random_indices:
    # source sentence
    en_sentence = " ".join([dataset.en_id2word[w] for w in dataset.dev_en_ids[idx]])
    print(f"\n{en_sentence}")

    # target sentence
    cn_sentence = " ".join([dataset.cn_id2word[w] for w in dataset.dev_cn_ids[idx]])
    print(cn_sentence)

    # to tensor
    src = torch.tensor([dataset.dev_en_ids[idx]], dtype=torch.long, device=dataset.device)

    # encoder padding mask
    src_mask = (src != PAD).unsqueeze(1).unsqueeze(2)

    # greedy decode
    """
    out = greedy_decode(model, src, src_mask, max_len=50, start_symbol=dataset.cn_vocab["BOS"])

    # prediction ids -> tokens
    pred_tokens = []

    for token_id in out[0, 1:]:
      token = dataset.cn_id2word[token_id.item()]
      if token == "EOS":
        break
      pred_tokens.append(token)

    print(" ".join(pred_tokens))
    """

In [ ]:
evaluate(dataset=data, model=model, num_examples=10)